In [1]:

#Backpropagation Implementation for Multi-Layer Perceptron (MLP) in Python
import math

# ---------- Sigmoid and derivative ----------
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def sigmoid_derivative(y):
    # y is sigmoid(x)
    return y * (1 - y)

# ---------- Inputs ----------
x1, x2, x3 = 1, 0, 1
target = 1
eta = 0.1

# ---------- Initial Weights ----------
# Input -> Hidden
w14, w24, w34 = 0.2, 0.4, -0.5
w15, w25, w35 = -0.3, 0.1, 0.2

# Biases (Hidden)
theta4, theta5 = 0.4, 0.2

# Hidden -> Output
w46, w56 = -0.3, -0.2
theta6 = 0.1

# ==================================================
# FORWARD PASS
# ==================================================

# Hidden layer
net_h4 = x1*w14 + x2*w24 + x3*w34 + theta4
h4 = sigmoid(net_h4)

net_h5 = x1*w15 + x2*w25 + x3*w35 + theta5
h5 = sigmoid(net_h5)

# Output layer
net_o6 = h4*w46 + h5*w56 + theta6
y6 = sigmoid(net_o6)

print("Forward Pass Output:", y6)

# ==================================================
# BACKPROPAGATION
# ==================================================

# Output error term
delta6 = (target - y6) * sigmoid_derivative(y6)

# Hidden layer error terms
delta4 = sigmoid_derivative(h4) * (delta6 * w46)
delta5 = sigmoid_derivative(h5) * (delta6 * w56)

# ==================================================
# WEIGHT UPDATES
# ==================================================

# Hidden -> Output
w46 += eta * delta6 * h4
w56 += eta * delta6 * h5

# Input -> Hidden (H4)
w14 += eta * delta4 * x1
w24 += eta * delta4 * x2
w34 += eta * delta4 * x3

# Input -> Hidden (H5)
w15 += eta * delta5 * x1
w25 += eta * delta5 * x2
w35 += eta * delta5 * x3

# ==================================================
# FORWARD PASS AGAIN (After update)
# ==================================================

net_h4 = x1*w14 + x2*w24 + x3*w34 + theta4
h4 = sigmoid(net_h4)

net_h5 = x1*w15 + x2*w25 + x3*w35 + theta5
h5 = sigmoid(net_h5)

net_o6 = h4*w46 + h5*w56 + theta6
y6_new = sigmoid(net_o6)

print("Updated Output:", y6_new)

Forward Pass Output: 0.45946674519334196
Updated Output: 0.4613575324422754


In [14]:
#a.backpropagation using numpy
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target.reshape(-1, 1)

# Normalize data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Sigmoid function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Derivative of sigmoid
def sigmoid_derivative(x):
    return x * (1 - x)

# Initialize weights
input_layer = X_train.shape[1]
hidden_layer = 16
output_layer = 1

np.random.seed(1)
W1 = np.random.randn(input_layer, hidden_layer)
b1 = np.zeros((1, hidden_layer))
W2 = np.random.randn(hidden_layer, output_layer)
b2 = np.zeros((1, output_layer))

learning_rate = 0.01
epochs = 1000

# Training
for epoch in range(epochs):
    # Forward propagation
    z1 = np.dot(X_train, W1) + b1
    a1 = sigmoid(z1)

    z2 = np.dot(a1, W2) + b2
    y_pred = sigmoid(z2)

    # Loss (Mean Squared Error)
    loss = np.mean((y_train - y_pred) ** 2)

    # Backpropagation
    d_output = (y_train - y_pred) * sigmoid_derivative(y_pred)
    d_hidden = np.dot(d_output, W2.T) * sigmoid_derivative(a1)

    # Update weights and biases
    W2 += np.dot(a1.T, d_output) * learning_rate
    b2 += np.sum(d_output, axis=0, keepdims=True) * learning_rate

    W1 += np.dot(X_train.T, d_hidden) * learning_rate
    b1 += np.sum(d_hidden, axis=0, keepdims=True) * learning_rate

    if epoch % 100 == 0:
        print("Epoch:", epoch, "Loss:", loss)

# Testing
z1 = np.dot(X_test, W1) + b1
a1 = sigmoid(z1)
z2 = np.dot(a1, W2) + b2
y_test_pred = sigmoid(z2)

y_test_pred_class = (y_test_pred > 0.5).astype(int)

accuracy = np.mean(y_test_pred_class == y_test)
print("Test Accuracy (NumPy):", accuracy)


Epoch: 0 Loss: 0.4214081763895656
Epoch: 100 Loss: 0.018797932931252256
Epoch: 200 Loss: 0.013646516988299846
Epoch: 300 Loss: 0.011366477581680889
Epoch: 400 Loss: 0.009872186192259364
Epoch: 500 Loss: 0.008674420880014626
Epoch: 600 Loss: 0.0076727977969793075
Epoch: 700 Loss: 0.006859484822730042
Epoch: 800 Loss: 0.00621043597253897
Epoch: 900 Loss: 0.005695739604557932
Test Accuracy (NumPy): 0.9824561403508771


In [17]:
#b.backpropagation using keras
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Normalize data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build Neural Network Model
model = Sequential()
model.add(Dense(16, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model
model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=1)

# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy (Keras):", accuracy)


Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.3450 - loss: 0.7528
Epoch 2/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7034 - loss: 0.6211
Epoch 3/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8257 - loss: 0.5205
Epoch 4/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8744 - loss: 0.4084
Epoch 5/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8968 - loss: 0.2831
Epoch 6/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9275 - loss: 0.2206
Epoch 7/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9416 - loss: 0.1919
Epoch 8/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9447 - loss: 0.1661
Epoch 9/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9563 - loss: 0.1425
Epoch 10/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9765 - loss: 0.1115 
Epoch 11/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9793 - loss: 0.0925 
Epoch 12/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9827 - loss: 0.088